In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Labs"
)

xpt_file = DATA_DIR / "GLU_G.xpt"
csv_file = DATA_DIR / "GLU_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Preserve completely untouched pandas-decoded XPT.
source_df = df.copy()


print("=" * 100)
print("NHANES 2011-2012 GLU_G XPT -> CSV")
print("=" * 100)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXGLU",
    "LBDGLUSI",
    "LBXIN",
    "LBDINSI",
    "PHAFSTHR",
    "PHAFSTMN"
]


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "GLU_G column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 3239:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 8:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 3,239 rows x 8 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# GLU_G is a MIXED whole-number / continuous-decimal file.
#
#
# WHOLE-NUMBER:
#
#     SEQN
#     LBXGLU
#     PHAFSTHR
#     PHAFSTMN
#
#
# GENUINE CONTINUOUS DECIMAL:
#
#     WTSAF2YR
#     LBDGLUSI
#     LBXIN
#     LBDINSI
#
#
# IMPORTANT:
#
# DO NOT:
#
#     round continuous variables
#     globally remove ".0"
#     use float_format=
#     convert genuine decimals to Int64
#
# Genuine decimal fields must remain AS IS.
#

DECIMAL_COLUMNS = [
    "WTSAF2YR",
    "LBDGLUSI",
    "LBXIN",
    "LBDINSI"
]


INTEGER_COLUMNS = [
    "SEQN",
    "LBXGLU",
    "PHAFSTHR",
    "PHAFSTMN"
]


if len(DECIMAL_COLUMNS) != 4:
    raise ValueError(
        "Expected exactly 4 genuine decimal fields."
    )


if len(INTEGER_COLUMNS) != 4:
    raise ValueError(
        "Expected exactly 4 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
# Never globally replace this before validating the
# complete variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:
    raise ValueError(
        "Expected all 8 GLU_G fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0).sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached GLU_G.xpt:
#
#     WTSAF2YR     397
#     PHAFSTHR      36
#     PHAFSTMN      54
#
# TOTAL            487
#

EXPECTED_TINY_COUNTS = {
    "WTSAF2YR": 397,
    "PHAFSTHR": 36,
    "PHAFSTMN": 54
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated GLU_G attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 487:
    raise ValueError(
        f"Unexpected tiny-value total: {tiny_total:,}"
    )


if tiny_variable_count != 3:
    raise ValueError(
        "Expected tiny values in exactly 3 variables."
    )


if true_zero_total != 0:
    raise ValueError(
        "Unexpected ordinary zeros existed "
        "before tiny-value restoration."
    )


print(
    "\nPASS: Exact 487-value tiny-zero pattern "
    "across 3 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# NEVER:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = df[col].eq(
        TINY_VALUE
    )

    actual_count = int(
        mask.sum()
    )

    if actual_count != expected_count:
        raise ValueError(
            f"{col}: expected {expected_count:,} "
            f"tiny values, found {actual_count:,}."
        )

    df.loc[
        mask,
        col
    ] = 0

    correction_total += actual_count

    print(
        f"{col}: {actual_count:,} restored to 0"
    )


if correction_total != 487:
    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df is the exact corrected-XPT fidelity reference.
#
# The ONLY permitted numeric changes:
#
#     validated tiny value -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:
    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY to test whether observations
# have a fractional component.
#
# It does NOT modify the dataframe.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


detected_decimal_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:

        fractional_counts[col] = 0
        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )

    fractional_count = int(
        fractional_mask.sum()
    )

    fractional_counts[col] = fractional_count

    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )

    if fractional_count > 0:
        detected_decimal_columns.append(
            col
        )


if detected_decimal_columns != DECIMAL_COLUMNS:
    raise ValueError(
        "STOP: Genuine decimal-variable pattern differs.\n\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Actual:\n{detected_decimal_columns}"
    )


print(
    "\nPASS: Exactly 4 genuine decimal fields detected."
)


# ============================================================
# 11. DECIMAL FIELD DETAILS
# ============================================================

print(
    "\n--- CONTINUOUS DECIMAL FIELD DETAILS ---"
)


for col in DECIMAL_COLUMNS:

    s = df[col]

    nonmissing = int(
        s.notna().sum()
    )

    missing = int(
        s.isna().sum()
    )

    fractional = fractional_counts[col]

    whole_looking = (
        nonmissing
        -
        fractional
    )

    print(
        f"\n{col}"
    )

    print(
        "  Nonmissing:",
        f"{nonmissing:,}"
    )

    print(
        "  Missing:",
        f"{missing:,}"
    )

    print(
        "  Minimum:",
        repr(float(s.min()))
    )

    print(
        "  Maximum:",
        repr(float(s.max()))
    )

    print(
        "  Fractional observations:",
        f"{fractional:,}"
    )

    print(
        "  Whole-looking observations:",
        f"{whole_looking:,}"
    )


# ============================================================
# 12. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"].isna().sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"].duplicated().sum()
    )
)


if df["SEQN"].min() != 62161:
    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 71916:
    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(df["SEQN"].isna().sum()) != 0:
    raise ValueError(
        "Missing SEQN found."
    )


if int(df["SEQN"].duplicated().sum()) != 0:
    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 62161-71916 validated."
)


# ============================================================
# 13. VALIDATE WHOLE-NUMBER LAB / FASTING FIELDS
# ============================================================

for col in [
    "LBXGLU",
    "PHAFSTHR",
    "PHAFSTMN"
]:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if not np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    ).all():

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: LBXGLU, PHAFSTHR and PHAFSTMN "
    "contain only whole-number values."
)


# ============================================================
# 14. CONVERT PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# NO .round() during actual conversion.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 4 proven whole-number fields "
    "converted to nullable Int64 without rounding."
)


# ============================================================
# 15. CONFIRM CONTINUOUS FIELDS REMAIN FLOAT
# ============================================================

decimal_dtype_failures = []


for col in DECIMAL_COLUMNS:

    if not pd.api.types.is_float_dtype(
        df[col].dtype
    ):

        decimal_dtype_failures.append(
            col
        )


if decimal_dtype_failures:
    raise ValueError(
        "Continuous variables are no longer float:\n"
        + str(decimal_dtype_failures)
    )


print(
    "PASS: WTSAF2YR, LBDGLUSI, LBXIN and LBDINSI "
    "remain floating-point."
)


# ============================================================
# 16. WTSAF2YR VALIDATION
# ============================================================

#
# CDC codebook:
#
#     WTSAF2YR range = 0 to 521033.371
#     nonmissing     = 3,239
#
#
# Direct attached XPT:
#
#     positive weights = 2,842
#     zero weights     =   397
#
#     positive min = 10219.081
#     max          = 521033.371
#

weight_positive = int(
    df["WTSAF2YR"]
    .gt(0)
    .sum()
)


weight_zero = int(
    df["WTSAF2YR"]
    .eq(0)
    .sum()
)


weight_missing = int(
    df["WTSAF2YR"]
    .isna()
    .sum()
)


positive_weights = (
    df.loc[
        df["WTSAF2YR"] > 0,
        "WTSAF2YR"
    ]
)


weight_min = float(
    positive_weights.min()
)


weight_max = float(
    positive_weights.max()
)


print(
    "\n--- WTSAF2YR ---"
)

print(
    "Positive:",
    f"{weight_positive:,}"
)

print(
    "Zero:",
    f"{weight_zero:,}"
)

print(
    "Missing:",
    f"{weight_missing:,}"
)

print(
    "Exact positive minimum:",
    repr(weight_min)
)

print(
    "Exact maximum:",
    repr(weight_max)
)


if weight_positive != 2842:
    raise ValueError(
        "WTSAF2YR positive count differs."
    )


if weight_zero != 397:
    raise ValueError(
        "WTSAF2YR zero count differs."
    )


if weight_missing != 0:
    raise ValueError(
        "Unexpected missing WTSAF2YR."
    )


if weight_min != 10219.081:
    raise ValueError(
        "WTSAF2YR positive minimum differs."
    )


if weight_max != 521033.371:
    raise ValueError(
        "WTSAF2YR maximum differs."
    )


print(
    "PASS: WTSAF2YR validated."
)


# ============================================================
# 17. CDC CHECK - LBXGLU
# ============================================================

#
# Fasting glucose mg/dL:
#
#     39-382
#     nonmissing = 3,033
#     missing    =   206
#

glu_nonmissing = int(
    df["LBXGLU"].notna().sum()
)


glu_missing = int(
    df["LBXGLU"].isna().sum()
)


glu_range = int(
    df["LBXGLU"]
    .between(
        39,
        382
    )
    .sum()
)


glu_min = int(
    df["LBXGLU"].min()
)


glu_max = int(
    df["LBXGLU"].max()
)


print(
    "\n--- LBXGLU ---"
)

print(
    "Nonmissing:",
    f"{glu_nonmissing:,}"
)

print(
    "Missing:",
    f"{glu_missing:,}"
)

print(
    "Minimum:",
    glu_min
)

print(
    "Maximum:",
    glu_max
)


if glu_nonmissing != 3033:
    raise ValueError(
        "LBXGLU nonmissing count differs."
    )


if glu_missing != 206:
    raise ValueError(
        "LBXGLU missing count differs."
    )


if glu_range != 3033:
    raise ValueError(
        "LBXGLU contains values outside 39-382."
    )


if glu_min != 39:
    raise ValueError(
        "LBXGLU minimum differs."
    )


if glu_max != 382:
    raise ValueError(
        "LBXGLU maximum differs."
    )


print(
    "PASS: LBXGLU matches CDC range/count."
)


# ============================================================
# 18. CDC CHECK - LBDGLUSI
# ============================================================

#
# Fasting glucose mmol/L:
#
#     2.165-21.205
#     nonmissing = 3,033
#     missing    =   206
#

glusi_nonmissing = int(
    df["LBDGLUSI"].notna().sum()
)


glusi_missing = int(
    df["LBDGLUSI"].isna().sum()
)


glusi_range = int(
    df["LBDGLUSI"]
    .between(
        2.165,
        21.205
    )
    .sum()
)


glusi_min = float(
    df["LBDGLUSI"].min()
)


glusi_max = float(
    df["LBDGLUSI"].max()
)


print(
    "\n--- LBDGLUSI ---"
)

print(
    "Nonmissing:",
    f"{glusi_nonmissing:,}"
)

print(
    "Missing:",
    f"{glusi_missing:,}"
)

print(
    "Minimum:",
    repr(glusi_min)
)

print(
    "Maximum:",
    repr(glusi_max)
)


if glusi_nonmissing != 3033:
    raise ValueError(
        "LBDGLUSI nonmissing count differs."
    )


if glusi_missing != 206:
    raise ValueError(
        "LBDGLUSI missing count differs."
    )


if glusi_range != 3033:
    raise ValueError(
        "LBDGLUSI contains values outside "
        "2.165-21.205."
    )


if glusi_min != 2.165:
    raise ValueError(
        "LBDGLUSI minimum differs."
    )


if glusi_max != 21.205:
    raise ValueError(
        "LBDGLUSI maximum differs."
    )


print(
    "PASS: LBDGLUSI matches CDC range/count."
)


# ============================================================
# 19. GLUCOSE MISSING-MASK CHECK
# ============================================================

glucose_mask_mismatches = int(
    np.sum(
        df["LBXGLU"]
        .isna()
        .to_numpy()
        !=
        df["LBDGLUSI"]
        .isna()
        .to_numpy()
    )
)


print(
    "\n--- GLUCOSE MISSING MASK ---"
)

print(
    "Mismatches:",
    glucose_mask_mismatches
)


if glucose_mask_mismatches != 0:
    raise ValueError(
        "LBXGLU and LBDGLUSI missing "
        "positions differ."
    )


print(
    "PASS: Glucose conventional/SI missing "
    "positions match exactly."
)


# ============================================================
# 20. CDC GLUCOSE CONVERSION CHECK
# ============================================================

#
# CDC:
#
#     LBDGLUSI = LBXGLU * 0.05551
#
# rounded to 3 decimal places.
#
#
# Validation ONLY.
#
# We NEVER overwrite the released LBDGLUSI.
#

valid_glucose_mask = (
    df["LBXGLU"].notna()
    &
    df["LBDGLUSI"].notna()
)


glucose_conversion_mismatches = 0


for mg_dl, released_si in zip(
    df.loc[
        valid_glucose_mask,
        "LBXGLU"
    ],
    df.loc[
        valid_glucose_mask,
        "LBDGLUSI"
    ]
):

    calculated = (
        Decimal(
            str(int(mg_dl))
        )
        *
        Decimal("0.05551")
    )


    calculated_3dp = calculated.quantize(
        Decimal("0.001"),
        rounding=ROUND_HALF_UP
    )


    released_3dp = Decimal(
        str(float(released_si))
    ).quantize(
        Decimal("0.001")
    )


    if calculated_3dp != released_3dp:
        glucose_conversion_mismatches += 1


print(
    "\n--- GLUCOSE SI CONVERSION CHECK ---"
)

print(
    "Records checked:",
    f"{int(valid_glucose_mask.sum()):,}"
)

print(
    "Conversion mismatches:",
    glucose_conversion_mismatches
)


if glucose_conversion_mismatches != 0:
    raise ValueError(
        "LBDGLUSI does not match documented "
        "LBXGLU x 0.05551 conversion."
    )


print(
    "PASS: All 3,033 glucose SI results "
    "match the documented conversion."
)


# ============================================================
# 21. CDC CHECK - LBXIN
# ============================================================

#
# Insulin uU/mL:
#
#     0.14-647.5
#     nonmissing = 2,881
#     missing    =   358
#

ins_nonmissing = int(
    df["LBXIN"].notna().sum()
)


ins_missing = int(
    df["LBXIN"].isna().sum()
)


ins_range = int(
    df["LBXIN"]
    .between(
        0.14,
        647.5
    )
    .sum()
)


ins_min = float(
    df["LBXIN"].min()
)


ins_max = float(
    df["LBXIN"].max()
)


print(
    "\n--- LBXIN ---"
)

print(
    "Nonmissing:",
    f"{ins_nonmissing:,}"
)

print(
    "Missing:",
    f"{ins_missing:,}"
)

print(
    "Minimum:",
    repr(ins_min)
)

print(
    "Maximum:",
    repr(ins_max)
)


if ins_nonmissing != 2881:
    raise ValueError(
        "LBXIN nonmissing count differs."
    )


if ins_missing != 358:
    raise ValueError(
        "LBXIN missing count differs."
    )


if ins_range != 2881:
    raise ValueError(
        "LBXIN contains values outside "
        "0.14-647.5."
    )


if ins_min != 0.14:
    raise ValueError(
        "LBXIN minimum differs."
    )


if ins_max != 647.5:
    raise ValueError(
        "LBXIN maximum differs."
    )


print(
    "PASS: LBXIN matches CDC range/count."
)


# ============================================================
# 22. CDC CHECK - LBDINSI
# ============================================================

#
# Insulin pmol/L:
#
#     0.85-3885
#     nonmissing = 2,881
#     missing    =   358
#

insi_nonmissing = int(
    df["LBDINSI"].notna().sum()
)


insi_missing = int(
    df["LBDINSI"].isna().sum()
)


insi_range = int(
    df["LBDINSI"]
    .between(
        0.85,
        3885
    )
    .sum()
)


insi_min = float(
    df["LBDINSI"].min()
)


insi_max = float(
    df["LBDINSI"].max()
)


print(
    "\n--- LBDINSI ---"
)

print(
    "Nonmissing:",
    f"{insi_nonmissing:,}"
)

print(
    "Missing:",
    f"{insi_missing:,}"
)

print(
    "Minimum:",
    repr(insi_min)
)

print(
    "Maximum:",
    repr(insi_max)
)


if insi_nonmissing != 2881:
    raise ValueError(
        "LBDINSI nonmissing count differs."
    )


if insi_missing != 358:
    raise ValueError(
        "LBDINSI missing count differs."
    )


if insi_range != 2881:
    raise ValueError(
        "LBDINSI contains values outside "
        "0.85-3885."
    )


if insi_min != 0.85:
    raise ValueError(
        "LBDINSI minimum differs."
    )


if insi_max != 3885.0:
    raise ValueError(
        "LBDINSI maximum differs."
    )


print(
    "PASS: LBDINSI matches CDC range/count."
)


# ============================================================
# 23. INSULIN MISSING-MASK CHECK
# ============================================================

insulin_mask_mismatches = int(
    np.sum(
        df["LBXIN"]
        .isna()
        .to_numpy()
        !=
        df["LBDINSI"]
        .isna()
        .to_numpy()
    )
)


print(
    "\n--- INSULIN MISSING MASK ---"
)

print(
    "Mismatches:",
    insulin_mask_mismatches
)


if insulin_mask_mismatches != 0:
    raise ValueError(
        "LBXIN and LBDINSI missing positions differ."
    )


print(
    "PASS: Insulin conventional/SI missing "
    "positions match exactly."
)


# ============================================================
# 24. CDC INSULIN CONVERSION CHECK
# ============================================================

#
# CDC documentation:
#
#     LBDINSI = LBXIN * 6.0
#
# rounded to 2 decimal places.
#
#
# IMPORTANT FIDELITY DETAIL:
#
# Recalculating SI from the ALREADY RELEASED/ROUNDED
# LBXIN values produces ONE expected apparent mismatch:
#
#     LBXIN   = 0.14
#     LBDINSI = 0.85
#
# because:
#
#     0.14 * 6 = 0.84
#
# but the SI value may have been generated using
# greater underlying source precision before LBXIN
# itself was released/rounded.
#
# Therefore we VALIDATE the pattern but NEVER regenerate
# or overwrite LBDINSI.
#

valid_insulin_mask = (
    df["LBXIN"].notna()
    &
    df["LBDINSI"].notna()
)


insulin_conversion_mismatches = []



for idx, insulin, released_si in zip(
    df.index[valid_insulin_mask],
    df.loc[
        valid_insulin_mask,
        "LBXIN"
    ],
    df.loc[
        valid_insulin_mask,
        "LBDINSI"
    ]
):

    calculated = (
        Decimal(
            str(float(insulin))
        )
        *
        Decimal("6.0")
    )


    calculated_2dp = calculated.quantize(
        Decimal("0.01"),
        rounding=ROUND_HALF_UP
    )


    released_2dp = Decimal(
        str(float(released_si))
    ).quantize(
        Decimal("0.01")
    )


    if calculated_2dp != released_2dp:

        insulin_conversion_mismatches.append(
            {
                "row": int(idx),
                "LBXIN": float(insulin),
                "LBDINSI": float(released_si),
                "calculated": str(calculated_2dp),
                "released": str(released_2dp)
            }
        )


print(
    "\n--- INSULIN SI CONVERSION CHECK ---"
)

print(
    "Records checked:",
    f"{int(valid_insulin_mask.sum()):,}"
)

print(
    "Apparent conversion mismatches:",
    len(insulin_conversion_mismatches)
)


if insulin_conversion_mismatches:

    print(
        insulin_conversion_mismatches
    )


if len(insulin_conversion_mismatches) != 1:
    raise ValueError(
        "Unexpected insulin conversion mismatch count."
    )


expected_insulin_mismatch = (
    insulin_conversion_mismatches[0]
)


if (
    expected_insulin_mismatch["LBXIN"] != 0.14
    or
    expected_insulin_mismatch["LBDINSI"] != 0.85
):
    raise ValueError(
        "Unexpected insulin conversion mismatch pattern."
    )


print(
    "PASS: Expected single lower-end insulin "
    "rounding discrepancy documented."
)

print(
    "PASS: Released LBXIN/LBDINSI values will "
    "remain untouched."
)


# ============================================================
# 25. CDC CHECK - PHAFSTHR
# ============================================================

#
# Total length of food fast, hours:
#
#     0-36
#     nonmissing = 3,167
#     missing    =    72
#
# Direct attachment zero count:
#
#     36
#

fast_hr_nonmissing = int(
    df["PHAFSTHR"].notna().sum()
)


fast_hr_missing = int(
    df["PHAFSTHR"].isna().sum()
)


fast_hr_range = int(
    df["PHAFSTHR"]
    .between(
        0,
        36
    )
    .sum()
)


fast_hr_zero = int(
    df["PHAFSTHR"]
    .eq(0)
    .sum()
)


print(
    "\n--- PHAFSTHR ---"
)

print(
    "Nonmissing:",
    f"{fast_hr_nonmissing:,}"
)

print(
    "Missing:",
    f"{fast_hr_missing:,}"
)

print(
    "Zero:",
    f"{fast_hr_zero:,}"
)


if fast_hr_nonmissing != 3167:
    raise ValueError(
        "PHAFSTHR nonmissing count differs."
    )


if fast_hr_missing != 72:
    raise ValueError(
        "PHAFSTHR missing count differs."
    )


if fast_hr_range != 3167:
    raise ValueError(
        "PHAFSTHR contains values outside 0-36."
    )


if fast_hr_zero != 36:
    raise ValueError(
        "PHAFSTHR zero count differs."
    )


print(
    "PASS: PHAFSTHR validated."
)


# ============================================================
# 26. CDC CHECK - PHAFSTMN
# ============================================================

#
# Total length of food fast, minutes:
#
#     0-59
#     nonmissing = 3,167
#     missing    =    72
#
# Direct attachment zero count:
#
#     54
#

fast_mn_nonmissing = int(
    df["PHAFSTMN"].notna().sum()
)


fast_mn_missing = int(
    df["PHAFSTMN"].isna().sum()
)


fast_mn_range = int(
    df["PHAFSTMN"]
    .between(
        0,
        59
    )
    .sum()
)


fast_mn_zero = int(
    df["PHAFSTMN"]
    .eq(0)
    .sum()
)


print(
    "\n--- PHAFSTMN ---"
)

print(
    "Nonmissing:",
    f"{fast_mn_nonmissing:,}"
)

print(
    "Missing:",
    f"{fast_mn_missing:,}"
)

print(
    "Zero:",
    f"{fast_mn_zero:,}"
)


if fast_mn_nonmissing != 3167:
    raise ValueError(
        "PHAFSTMN nonmissing count differs."
    )


if fast_mn_missing != 72:
    raise ValueError(
        "PHAFSTMN missing count differs."
    )


if fast_mn_range != 3167:
    raise ValueError(
        "PHAFSTMN contains values outside 0-59."
    )


if fast_mn_zero != 54:
    raise ValueError(
        "PHAFSTMN zero count differs."
    )


print(
    "PASS: PHAFSTMN validated."
)


# ============================================================
# 27. FASTING TIME MISSING-MASK CHECK
# ============================================================

fasting_mask_mismatches = int(
    np.sum(
        df["PHAFSTHR"]
        .isna()
        .to_numpy()
        !=
        df["PHAFSTMN"]
        .isna()
        .to_numpy()
    )
)


print(
    "\n--- FASTING TIME MISSING MASK ---"
)

print(
    "Mismatches:",
    fasting_mask_mismatches
)


if fasting_mask_mismatches != 0:
    raise ValueError(
        "PHAFSTHR and PHAFSTMN missing "
        "positions differ."
    )


print(
    "PASS: Fasting hour/minute missing "
    "positions match exactly."
)


# ============================================================
# 28. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original pandas-decoded XPT contained no ordinary zeros.
#
# Therefore all zeros after correction must exactly match
# the validated tiny-value positions.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )

    if zero_count > 0:
        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "Final zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 487:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 487 validated restorations."
)


# ============================================================
# 29. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY numeric modifications permitted:
#
#     validated tiny -> 0
#
# Everything else must remain numerically identical.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:
        pre_export_failures.append(
            col
        )


if pre_export_failures:
    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: All 8 GLU_G variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 30. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# Whole-number fields lose unnecessary ".0":
#
#     SEQN
#     LBXGLU
#     PHAFSTHR
#     PHAFSTMN
#
#
# Continuous fields remain float:
#
#     WTSAF2YR
#     LBDGLUSI
#     LBXIN
#     LBDINSI
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 31. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:
    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):
    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 32. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:
        csv_failures.append(
            col
        )


if csv_failures:
    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: 0 numeric differences across "
    "all 8 GLU_G variables."
)


# ============================================================
# 33. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:
    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 34. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 35. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:
    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 36. .0 CHECK - WHOLE-NUMBER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT check the continuous columns for ".0".
#
# Only:
#
#     SEQN
#     LBXGLU
#     PHAFSTHR
#     PHAFSTMN
#
# should have unnecessary ".0" removed.
#

print(
    "\n--- WHOLE-NUMBER FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]

    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )

    count = int(
        bad_mask.sum()
    )

    if count > 0:
        dot_zero_errors[col] = count


if dot_zero_errors:
    raise ValueError(
        "Unexpected '.0' in whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "SEQN, LBXGLU, PHAFSTHR or PHAFSTMN."
)


# ============================================================
# 37. LEGITIMATE CONTINUOUS .0 TOKENS
# ============================================================

#
# Direct tested export:
#
#     WTSAF2YR = 398 tokens ending .0
#     LBDGLUSI =   0
#     LBXIN    =  30
#     LBDINSI  =  65
#
#
# These are NOT errors.
#
# They correspond to genuine continuous columns whose
# numeric value happens to be whole-looking.
#

print(
    "\n--- LEGITIMATE CONTINUOUS .0 TOKENS ---"
)


EXPECTED_DECIMAL_DOT_ZERO_COUNTS = {
    "WTSAF2YR": 398,
    "LBDGLUSI": 0,
    "LBXIN": 30,
    "LBDINSI": 65
}


decimal_dot_zero_counts = {}


for col in DECIMAL_COLUMNS:

    count = int(
        (
            raw_csv_df[col].ne("")
            &
            raw_csv_df[col].str.endswith(".0")
        )
        .sum()
    )

    decimal_dot_zero_counts[col] = count

    print(
        f"{col}:",
        f"{count:,}"
    )


if (
    decimal_dot_zero_counts
    !=
    EXPECTED_DECIMAL_DOT_ZERO_COUNTS
):
    raise ValueError(
        "Continuous '.0' token pattern differs.\n\n"
        f"Expected:\n"
        f"{EXPECTED_DECIMAL_DOT_ZERO_COUNTS}\n\n"
        f"Actual:\n"
        f"{decimal_dot_zero_counts}"
    )


print(
    "PASS: Legitimate continuous '.0' "
    "values preserved."
)


# ============================================================
# 38. EXACT CONTINUOUS-VALUE FIDELITY
# ============================================================

#
# Most important continuous-value test:
#
#     rtol = 0
#     atol = 0
#
# No numerical difference is accepted.
#

print(
    "\n--- EXACT CONTINUOUS DECIMAL FIDELITY ---"
)


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:
        decimal_failures.append(
            col
        )


if decimal_failures:
    raise ValueError(
        "Continuous-value fidelity failed:\n"
        + str(decimal_failures)
    )


print(
    "PASS: WTSAF2YR, LBDGLUSI, LBXIN and "
    "LBDINSI preserved exactly."
)


# ============================================================
# 39. FINAL RANGE CHECK
# ============================================================

final_weight = pd.to_numeric(
    raw_csv_df["WTSAF2YR"]
    .where(
        raw_csv_df["WTSAF2YR"] != "",
        np.nan
    ),
    errors="coerce"
)


final_glu = pd.to_numeric(
    raw_csv_df["LBXGLU"]
    .where(
        raw_csv_df["LBXGLU"] != "",
        np.nan
    ),
    errors="coerce"
)


final_glusi = pd.to_numeric(
    raw_csv_df["LBDGLUSI"]
    .where(
        raw_csv_df["LBDGLUSI"] != "",
        np.nan
    ),
    errors="coerce"
)


final_ins = pd.to_numeric(
    raw_csv_df["LBXIN"]
    .where(
        raw_csv_df["LBXIN"] != "",
        np.nan
    ),
    errors="coerce"
)


final_insi = pd.to_numeric(
    raw_csv_df["LBDINSI"]
    .where(
        raw_csv_df["LBDINSI"] != "",
        np.nan
    ),
    errors="coerce"
)


if final_weight.min() != 0:
    raise ValueError(
        "Final WTSAF2YR minimum changed."
    )


if final_weight.max() != 521033.371:
    raise ValueError(
        "Final WTSAF2YR maximum changed."
    )


if final_glu.min() != 39:
    raise ValueError(
        "Final LBXGLU minimum changed."
    )


if final_glu.max() != 382:
    raise ValueError(
        "Final LBXGLU maximum changed."
    )


if final_glusi.min() != 2.165:
    raise ValueError(
        "Final LBDGLUSI minimum changed."
    )


if final_glusi.max() != 21.205:
    raise ValueError(
        "Final LBDGLUSI maximum changed."
    )


if final_ins.min() != 0.14:
    raise ValueError(
        "Final LBXIN minimum changed."
    )


if final_ins.max() != 647.5:
    raise ValueError(
        "Final LBXIN maximum changed."
    )


if final_insi.min() != 0.85:
    raise ValueError(
        "Final LBDINSI minimum changed."
    )


if final_insi.max() != 3885.0:
    raise ValueError(
        "Final LBDINSI maximum changed."
    )


print(
    "PASS: Final laboratory ranges preserved."
)


# ============================================================
# 40. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    zero_count = int(
        values.eq(0).sum()
    )

    if zero_count > 0:
        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "487 validated zero values."
)


# ============================================================
# 41. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:
    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 42. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 62161:
    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 71916:
    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:
    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):
    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 43. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "GLU_G CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2011-2012"
)


print(
    "Component: Laboratory - "
    "Plasma Fasting Glucose & Insulin"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Tiny artifact variables:",
    list(
        EXPECTED_TINY_COUNTS.keys()
    )
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


print(
    "WTSAF2YR positive:",
    f"{weight_positive:,}"
)


print(
    "WTSAF2YR zero:",
    f"{weight_zero:,}"
)


print(
    "WTSAF2YR positive minimum:",
    repr(weight_min)
)


print(
    "WTSAF2YR maximum:",
    repr(weight_max)
)


print(
    "LBXGLU nonmissing:",
    f"{glu_nonmissing:,}"
)


print(
    "LBXGLU missing:",
    f"{glu_missing:,}"
)


print(
    "LBXGLU range:",
    f"{glu_min}-{glu_max} mg/dL"
)


print(
    "LBDGLUSI range:",
    f"{glusi_min}-{glusi_max} mmol/L"
)


print(
    "LBXIN nonmissing:",
    f"{ins_nonmissing:,}"
)


print(
    "LBXIN missing:",
    f"{ins_missing:,}"
)


print(
    "LBXIN range:",
    f"{ins_min}-{ins_max} uU/mL"
)


print(
    "LBDINSI range:",
    f"{insi_min}-{insi_max} pmol/L"
)


print(
    "PHAFSTHR zero:",
    fast_hr_zero
)


print(
    "PHAFSTMN zero:",
    fast_mn_zero
)


print(
    "Glucose conversion mismatches:",
    glucose_conversion_mismatches
)


print(
    "Expected insulin rounding discrepancy:",
    len(
        insulin_conversion_mismatches
    )
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Continuous-value differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0' in whole-number fields:",
    "0 expected"
)


print(
    "Continuous '.0' values:",
    "ALLOWED / PRESERVED"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "WTSAF2YR, LBDGLUSI, LBXIN and LBDINSI "
    "remain genuine continuous decimal variables."
)


print(
    "No continuous Laboratory value was rounded "
    "or rewritten."
)

NHANES 2011-2012 GLU_G XPT -> CSV
Rows: 3,239
Columns: 8
PASS: Structure = 3,239 rows x 8 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['WTSAF2YR', 'LBDGLUSI', 'LBXIN', 'LBDINSI']
Whole-number fields: ['SEQN', 'LBXGLU', 'PHAFSTHR', 'PHAFSTMN']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 487
Variables containing tiny artifacts: 3

Tiny values by variable:
WTSAF2YR    397
PHAFSTMN     54
PHAFSTHR     36

PASS: Exact 487-value tiny-zero pattern across 3 variables validated.

--- VALIDATED ZERO RESTORATION ---
WTSAF2YR: 397 restored to 0
PHAFSTHR: 36 restored to 0
PHAFSTMN: 54 restored to 0

PASS: 487 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
WTSAF2YR: 2,841 fractional observations
LBXGLU: 0 fractional observations
LBDGLUSI: 3,033 fractional observations
LBXIN: 2,851 fractional observations
LBDINSI: 2,816 frac